# 1. LLM Gateway

*Estimated time to run notebook: about 5 min*

**Goal:** Establish the foundation for our agent by connecting to the **DataRobot LLM Gateway**.

**Key Concept:**
Instead of managing API keys for every provider (Azure, AWS Bedrock, Google Vertex), DataRobot provides a single, unified endpoint. In this notebook, we verify access to nearly 100 different LLMs and select a base model (`azure/gpt-5-1-2025-11-13` or similar) to power our agent's reasoning capabilities.

### Workshop `.env` at the repository root

*In case you haven't done it in* **Notebook 0 – MCP Server Setup** *already*: 

Ensure you have a **`.env` file at the workshop repository root** (same folder as this notebook, `pyproject.toml`, and `uv.lock`):

1. **In a terminal** (from that root directory), run:
   ```bash
   cp .env.example .env
   ```
2. **Open `.env`** in your editor to add  **your** DataRobot credentials and other variables.
3. **Save** the file.

Notebook **0 – MCP Server Setup** explains deploying an MCP server and setting **`MCP_DEPLOYMENT_ID`** in this same root `.env` for later notebooks.

In [1]:
# Install required packages from the checked-in uv.lock
#!uv export --format requirements.txt --locked --no-emit-project | uv pip install -q -r -
!uv pip install pydantic-ai -q

### Connect to DataRobot LLM Gateway and inspect number of available LLMs

In [2]:
import datarobot as dr
from pprint import pprint

# 1. Initialize client
# This uses your local/notebook DataRobot credentials.
dr_client = dr.Client()

# 2. Query the LLM Gateway catalog
response = dr_client.get(url="genai/llmgw/catalog/")

# 3. Extract supported model IDs
# The catalog returns a list of model metadata; we keep just the string identifiers.
data = response.json()["data"]
supported_llms = [llm_model["model"] for llm_model in data]

# 4. Inspect results
print("Number of LLMs supported by LLM Gateway:", len(supported_llms))
pprint(supported_llms)

Number of LLMs supported by LLM Gateway: 100
['azure/gpt-5-5-2026-04-23',
 'azure/gpt-5-4-2026-03-05',
 'azure/gpt-5-1-2025-11-13',
 'azure/gpt-5-codex-2025-09-15',
 'azure/gpt-5-2025-08-07',
 'azure/gpt-5-mini-2025-08-07',
 'azure/gpt-5-nano-2025-08-07',
 'azure/gpt-4o-2024-11-20',
 'azure/gpt-4o-mini',
 'azure/gpt-4-turbo',
 'azure/gpt-4-32k',
 'azure/o4-mini',
 'azure/gpt-4',
 'azure/gpt-35-turbo',
 'azure/o3',
 'azure/o3-mini',
 'azure/o1-mini',
 'azure/o1',
 'bedrock/amazon.nova-premier-v1:0',
 'bedrock/amazon.nova-pro-v1:0',
 'bedrock/amazon.nova-lite-v1:0',
 'bedrock/amazon.nova-micro-v1:0',
 'bedrock/amazon.titan-text-express-v1',
 'bedrock/anthropic.claude-opus-4-5-20251101-v1:0',
 'bedrock/anthropic.claude-sonnet-4-5-20250929-v1:0',
 'bedrock/anthropic.claude-opus-4-1-20250805-v1:0',
 'bedrock/anthropic.claude-opus-4-20250514-v1:0',
 'bedrock/anthropic.claude-sonnet-4-20250514-v1:0',
 'bedrock/anthropic.claude-3-7-sonnet-20250219-v1:0',
 'bedrock/anthropic.claude-3-5-sonnet-2

### Select a model to use the DataRobot LLM Gateway and ask a question

In [3]:
import os
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

# 1. Load configuration
# .env is optional; defaults are provided below.
load_dotenv()

# 2. Configure the model (via DataRobot LLM Gateway)
MODEL_NAME = os.getenv("MODEL_NAME")
llm = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token,
        base_url=dr_client.endpoint + "/genai/llmgw",
    ),
)

# 3. Define the agent
agent = Agent(model=llm)

# 4. Execution (sanity-check)
response = await agent.run("What is the capital of France?")
pprint(response.output)

'The capital of France is Paris.'


⚠️ Note on Kernel Limits: DataRobot Codespaces have a limit of 5 active notebook kernels at a time. To ensure a smooth transition to the next exercise, please remember to shut down this kernel (by closing the notebook tab) once you are finished. This prevents any 'limit reached' errors when opening subsequent notebooks!